# Silver Layer — Services

> **READINESS: needs profiling review before approval.**
> Nothing is published until `profiling_approved` is set to `true`.

## Source and target
`<source_schema>.services` -> `<target_schema>.services`
(defaults: `cdr_mimiciv.bronze` -> `cdr_mimiciv.silver`; set with the notebook widgets).

## Grain and key semantics (official MIMIC-IV hosp documentation)
The `services` table describes the clinical service a patient was admitted under (and
transferred between) during a hospital stay. A patient can be physically located in a unit
(for example the MICU) while being cared for by a different service, so this table, not
`transfers`, identifies the service of care.
- `subject_id` links to `patients`; `hadm_id` links to `admissions`.
- `transfertime` is when the patient moved from `prev_service` (if present) to `curr_service`.
- `prev_service` / `curr_service` are service abbreviations (`VARCHAR(20)`), stored exactly as in
  the hospital database. Documented values: CMED, CSURG, DENT, ENT, EYE, GU, GYN, MED, NB, NBB,
  NMED, NSURG, OBS, ORTHO, OMED, PSURG, PSYCH, SURG, TRAUM, TSURG, VSURG.
- The documentation does **not** declare a unique key. One row is one service transition
  within a hospital stay; candidate key `(hadm_id, transfertime)` is **unverified** and is
  profiled only, not asserted.

## Silver objective
Conservative, typed copy: all source columns unchanged in their Bronze types plus the
technical `_silver_processed_at`. No flags, normalisation, NULL filling or deduplication
until profiling justifies them. The relationship to `admissions` is observational only.

## Open profiling questions (to be answered by running the profiling cells)
1. Is `(hadm_id, transfertime)` unique? Are there exact duplicate rows or same-time ties?
2. Null counts for each column (`prev_service` is expected to be NULL for the first service).
3. Values of `curr_service` / `prev_service`: all in the documented list? case/whitespace variants?
4. Is `prev_service` consistent with the preceding row's `curr_service` within a `hadm_id`?
5. Does every `hadm_id` / `subject_id` exist in admissions / patients, and does `subject_id`
   agree with admissions for the same `hadm_id`?
6. How does `transfertime` fall relative to `admittime` / `dischtime`? Do all admissions have
   at least one service row?
7. Can `hadm_id` be NULL? (Gold fact filters `hadm_id IS NOT NULL`.)

## Publication
Published with `CREATE OR REPLACE` semantics. Gold `dim_service` and the service-transfer
fact depend on this table.

In [0]:
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")
dbutils.widgets.text("profiling_approved", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"
PROFILING_APPROVED = dbutils.widgets.get("profiling_approved").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

from pyspark.sql import functions as F

SOURCE_TABLE = f"{BRONZE}.services"
TARGET_TABLE = f"{SILVER}.services"

SOURCE_COLUMNS = [
    "subject_id",
    "hadm_id",
    "transfertime",
    "prev_service",
    "curr_service",
]

In [0]:
services_bronze = spark.table(SOURCE_TABLE)

services_bronze.printSchema()
display(services_bronze.limit(20))

root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- transfertime: timestamp (nullable = true)
 |-- prev_service: string (nullable = true)
 |-- curr_service: string (nullable = true)



subject_id,hadm_id,transfertime,prev_service,curr_service
10000032,22595853,2180-05-06T22:24:57.000Z,null,MED
10000032,22841357,2180-06-26T18:28:08.000Z,null,MED
10000032,25742920,2180-08-05T23:44:50.000Z,null,MED
10000032,29079034,2180-07-23T12:36:04.000Z,null,MED
10000068,25022803,2160-03-03T23:17:17.000Z,null,MED
10000084,23052089,2160-11-21T01:57:17.000Z,null,MED
10000084,29888819,2160-12-28T05:13:06.000Z,null,MED
10000108,27250926,2163-09-27T23:18:04.000Z,null,MED
10000117,22927623,2181-11-15T02:06:42.000Z,null,MED
10000117,27988844,2183-09-18T18:11:32.000Z,null,ORTHO


In [0]:
# establish the grain: rows vs candidate keys (uniqueness is NOT assumed)
display(
    services_bronze.select(
        F.count("*").alias("total_rows"),
        F.countDistinct("hadm_id", "transfertime").alias("distinct_hadm_transfertime"),
        F.countDistinct("hadm_id").alias("distinct_hadm_id"),
        F.countDistinct("subject_id").alias("distinct_subject_id"),
    )
)

total_rows,distinct_hadm_transfertime,distinct_hadm_id,distinct_subject_id
593071,593070,546012,223448


%md
### Same-timestamp service transitions

Profiling identified **one repeated `(hadm_id, transfertime)` pair**, involving two distinct service-transition records. No exact duplicate rows were found.

Review of the surrounding records suggests this may reflect a clerical correction or an immediate service reassignment. The cause cannot be confirmed from the available data.

**Decision:** Retain both records unchanged. No deduplication or timestamp correction is applied. Treat this as a documented observation rather than a blocking error; the order of transitions sharing that timestamp remains ambiguous.

In [0]:
# duplicates: by candidate key, and across all source columns
duplicate_keys = (
    services_bronze.groupBy("hadm_id", "transfertime").count().filter(F.col("count") > 1)
)
exact_duplicate_rows = (
    services_bronze.groupBy(*SOURCE_COLUMNS).count().filter(F.col("count") > 1)
)

print("Duplicate (hadm_id, transfertime) groups:", duplicate_keys.count())
print("Exact duplicate row groups:", exact_duplicate_rows.count())
display(duplicate_keys.orderBy(F.desc("count")).limit(20))

Duplicate (hadm_id, transfertime) groups: 1
Exact duplicate row groups: 0


hadm_id,transfertime,count
21555795,2111-01-20T15:31:57.000Z,2


In [0]:
%sql 
SELECT * 
FROM cdr_mimiciv.bronze.services
WHERE hadm_id = 21555795

subject_id,hadm_id,transfertime,prev_service,curr_service
10822967,21555795,2111-01-20T15:23:56.000Z,null,SURG
10822967,21555795,2111-01-20T15:31:57.000Z,OMED,SURG
10822967,21555795,2111-01-20T15:31:57.000Z,SURG,OMED
10822967,21555795,2111-01-20T15:34:41.000Z,SURG,OMED


In [0]:
# profile missingness
display(
    services_bronze.select(
        *[F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in SOURCE_COLUMNS]
    )
)

subject_id,hadm_id,transfertime,prev_service,curr_service
0,0,0,544365,0


In [0]:
# categorical values (including case / whitespace variants)
display(services_bronze.groupBy("curr_service").count().orderBy(F.desc("count")))
display(services_bronze.groupBy("prev_service").count().orderBy(F.desc("count")))

display(
    services_bronze.select(
        F.countDistinct("curr_service").alias("distinct_curr"),
        F.countDistinct(F.upper(F.trim("curr_service"))).alias("distinct_curr_normalised"),
        F.countDistinct("prev_service").alias("distinct_prev"),
        F.countDistinct(F.upper(F.trim("prev_service"))).alias("distinct_prev_normalised"),
        F.sum(F.when(F.col("prev_service") == F.col("curr_service"), 1).otherwise(0)).alias("prev_equals_curr"),
    )
)

curr_service,count
MED,286528
CMED,54452
SURG,51640
OMED,33549
ORTHO,26991
NMED,26709
OBS,24923
NSURG,16795
CSURG,13421
VSURG,12385


prev_service,count
null,544365
MED,19714
SURG,5945
CMED,5498
OMED,3646
NMED,3087
ORTHO,2282
NSURG,2227
TRAUM,1714
TSURG,1061


distinct_curr,distinct_curr_normalised,distinct_prev,distinct_prev_normalised,prev_equals_curr
21,21,20,20,0


In [0]:
# sequence consistency within a hospital stay (observational)
from pyspark.sql import Window

_w = Window.partitionBy("hadm_id").orderBy("transfertime")
sequence = (
    services_bronze
    .withColumn("_row_in_stay", F.row_number().over(_w))
    .withColumn("_prior_curr_service", F.lag("curr_service").over(_w))
)

display(
    sequence.select(
        F.sum(F.when((F.col("_row_in_stay") == 1) & F.col("prev_service").isNotNull(), 1).otherwise(0)).alias("first_row_with_prev_service"),
        F.sum(F.when((F.col("_row_in_stay") > 1) & F.col("prev_service").isNull(), 1).otherwise(0)).alias("later_row_without_prev_service"),
        F.sum(F.when(
            (F.col("_row_in_stay") > 1) & F.col("prev_service").isNotNull()
            & (F.col("prev_service") != F.col("_prior_curr_service")), 1
        ).otherwise(0)).alias("prev_service_breaks_chain"),
    )
)

first_row_with_prev_service,later_row_without_prev_service,prev_service_breaks_chain
1913,266,3


In [0]:
display(sequence)

  File <command-7133797223268731>, line 1
    display(sequence) 20169791
                      ^
SyntaxError: invalid syntax


In [0]:
# relationships to the Silver reference tables (observational only; not row-level flags)
admissions_ref = spark.table(f"{SILVER}.admissions").select("hadm_id", "subject_id", "admittime", "dischtime")
patients_ref = spark.table(f"{SILVER}.patients").select("subject_id")

orphan_hadm = (
    services_bronze.filter(F.col("hadm_id").isNotNull())
    .join(admissions_ref.select("hadm_id"), "hadm_id", "left_anti")
    .count()
)
orphan_subject = services_bronze.join(patients_ref, "subject_id", "left_anti").count()

linked = services_bronze.alias("s").join(
    admissions_ref.alias("a"), F.col("s.hadm_id") == F.col("a.hadm_id"), "inner"
)

display(
    linked.select(
        F.sum(F.when(F.col("s.subject_id") != F.col("a.subject_id"), 1).otherwise(0)).alias("subject_id_differs_from_admissions"),
        F.sum(F.when(F.col("s.transfertime") < F.col("a.admittime"), 1).otherwise(0)).alias("transfertime_before_admittime"),
        F.sum(F.when(F.col("s.transfertime") > F.col("a.dischtime"), 1).otherwise(0)).alias("transfertime_after_dischtime"),
    )
)

admissions_without_service = (
    admissions_ref.select("hadm_id").distinct()
    .join(services_bronze.select("hadm_id").distinct(), "hadm_id", "left_anti")
    .count()
)

print("hadm_id not found in admissions:", orphan_hadm)
print("subject_id not found in patients:", orphan_subject)
print("admissions with no services row:", admissions_without_service)

subject_id_differs_from_admissions,transfertime_before_admittime,transfertime_after_dischtime
0,27595,531


hadm_id not found in admissions: 0
subject_id not found in patients: 0
admissions with no services row: 16


## Profiling Findings

_To be completed by the reviewer after running the profiling cells above. No findings
have been recorded yet; set `profiling_approved` to `true` only after review._

In [0]:
# conservative transformation: every source column unchanged in its Bronze type
services_silver = (
    services_bronze
    .select(*SOURCE_COLUMNS)
    .withColumn("_silver_processed_at", F.current_timestamp())
)

services_silver.printSchema()

root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- transfertime: timestamp (nullable = true)
 |-- prev_service: string (nullable = true)
 |-- curr_service: string (nullable = true)
 |-- _silver_processed_at: timestamp (nullable = false)



In [0]:
# pre-publish validation contract
# types follow the official documentation (INT, INT, TIMESTAMP, VARCHAR, VARCHAR)
expected_schema = {
    "subject_id": "int",
    "hadm_id": "int",
    "transfertime": "timestamp",
    "prev_service": "string",
    "curr_service": "string",
    "_silver_processed_at": "timestamp",
}
prepared_schema = {f.name: f.dataType.simpleString() for f in services_silver.schema.fields}

bronze_rows = services_bronze.count()
checks = services_silver.agg(
    F.count("*").alias("silver_rows"),
    F.count_if(F.col("subject_id").isNull()).alias("null_subject_id"),
    F.count_if(F.col("_silver_processed_at").isNull()).alias("missing_timestamps"),
).first().asDict()

print("Bronze rows:", bronze_rows)
print(checks)
print("Prepared schema:", prepared_schema)

assert checks["silver_rows"] == bronze_rows, "Silver transformation changed the row count"
assert checks["missing_timestamps"] == 0
assert list(prepared_schema.items()) == list(expected_schema.items()), prepared_schema
# Key uniqueness of (hadm_id, transfertime) is NOT established by the documentation;
# it is profiled above and deliberately not asserted.

Bronze rows: 593071
{'silver_rows': 593071, 'null_subject_id': 0, 'missing_timestamps': 0}
Prepared schema: {'subject_id': 'int', 'hadm_id': 'int', 'transfertime': 'timestamp', 'prev_service': 'string', 'curr_service': 'string', '_silver_processed_at': 'timestamp'}


In [0]:
# publish gate: profiling must be reviewed first
if not PROFILING_APPROVED:
    raise RuntimeError(
        "services is not published: profiling has not been reviewed. "
        "Run the profiling cells, record the findings, then re-run with profiling_approved=true."
    )

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

(
    services_silver.writeTo(TARGET_TABLE)
    .using("delta")
    .createOrReplace()
)

print(f"Published: {TARGET_TABLE}")

Published: cdr_mimiciv.silver.services


In [0]:
# post-publish validation (services is a small table; exceptAll is acceptable)
services_published = spark.table(TARGET_TABLE)
published_schema = {f.name: f.dataType.simpleString() for f in services_published.schema.fields}

published_rows = services_published.count()
lost = services_bronze.select(*SOURCE_COLUMNS).exceptAll(services_published.select(*SOURCE_COLUMNS)).count()
added = services_published.select(*SOURCE_COLUMNS).exceptAll(services_bronze.select(*SOURCE_COLUMNS)).count()
missing_ts = services_published.filter(F.col("_silver_processed_at").isNull()).count()

print("Published rows:", published_rows, "lost:", lost, "added:", added)
print("Schema:", published_schema)

assert published_rows == services_bronze.count()
assert lost == 0 and added == 0
assert missing_ts == 0
assert list(published_schema.items()) == list(expected_schema.items())

dbutils.notebook.exit("services:OK")